# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [20]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [21]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [22]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"# Enter dataset path
GENRES = sorted(os.listdir(os.path.join(DATA_ROOT, "genres_stems"))) # Make the list of all genres available (alphabetical order)
STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"] # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [34]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------
    total_corrupted = 0
    total_less_50491 = 0
    total_greater_50493 = 0

    for genre in GENRES:
        # Iterate through Genres
        genre_path = os.path.join(root_dir, "genres_stems", genre)
        # Check: if genre folder exists
        if not os.path.exists(genre_path):
            continue
        # CHECK : Completeness (Does it have all stems?)
        songs = sorted(os.listdir(genre_path))
        valid_songs = []
        for song in songs:
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue
            if not all(os.path.exists(os.path.join(song_path, stem)) for stem in STEMS):
                continue
            corrupted = False
        # CHECK : Corruption (Is any file too small? (less than 4kb))
        # size checks
            for stem in STEMS:
                file_path = os.path.join(song_path, stem)
                size = os.path.getsize(file_path)

                # < 4KB corruption
                if size < 4 * 1024:
                    total_corrupted += 1
                    corrupted = True

                # Size conditions
                if size < 5.0491 * 1024 * 1024:
                    total_less_50491 += 1

                if size > 5.0493 * 1024 * 1024:
                    total_greater_50493 += 1

            if not corrupted:
                valid_songs.append(song_path)

        rng.shuffle(valid_songs)

        split_idx = int(len(valid_songs) * (1 - val_split))

        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]
        # Stratified Shuffle Split
     #-------------------------------------------------------------------------

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            for song_path in song_list:
                for stem in STEMS:
                    stem_name = stem.replace(".wav", "")
                    target_dict[genre][stem_name].append(
                        os.path.join(song_path, stem)
                    )

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    print("Q1:", total_corrupted + total_less_50491)
    print("Q2:", abs(total_greater_50493 - total_less_50491))
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

Q1: 1256
Q2: 1072


In [35]:
print("Q3:",abs(len(tr["reggae"]["drums"]) -len(val["country"]["vocals"])))

Q3: 66


In [37]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """

    records = []

    # ------------------- write your code here -------------------------------

    total_files = 0  # ---- COUNT TOTAL FILES ----

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:

                total_files += 1

                # ------------------- Load Audio -------------------------------
                y, _ = librosa.load(file_path, sr=sr)
                total_duration = librosa.get_duration(y=y, sr=sr)

                # ------------------- Find Non-Silent Intervals ----------------
                intervals = librosa.effects.split(y, top_db=top_db)

                silence_segments = []
                prev_end = 0

                # CASE A: Fully silent
                if len(intervals) == 0:
                    silence_segments = [(0, total_duration)]

                else:
                    for start, end in intervals:
                        start_sec = start / sr
                        end_sec = end / sr

                        # Silence between segments
                        if start_sec > prev_end:
                            silence_segments.append((prev_end, start_sec))

                        prev_end = end_sec

                    # Check silence at end
                    if prev_end < total_duration:
                        silence_segments.append((prev_end, total_duration))

                max_silence = 0
                silence_location = ""

                # Determine max silence and its location
                for start, end in silence_segments:
                    silence_len = end - start

                    if silence_len > max_silence:
                        max_silence = silence_len

                        # CASE B: START silence
                        if start == 0:
                            silence_location = "START"

                        # CASE C: END silence
                        elif end == total_duration:
                            silence_location = "END"

                        # CASE D: MIDDLE silence
                        else:
                            silence_location = "MIDDLE"

                # ------------------- Store result -----------------------------
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": silence_location,
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records)


    print("Q4:", len(df))

    print("Q5:", len(df[df["Stem"] == "vocals"]))

    print("Q6:",round(df[df["Stem"] == "vocals"]["Max_Silence_Sec"].mean(), 2))

    print("Q7:",len(df[(df["Genre"] == "jazz") &(df["Stem"] == "drums")]))

    print("Q8:",len(df[(df["Genre"] == "jazz") &(df["Stem"] == "drums") &(df["Silence_Location"] == "MIDDLE")]))

    print("Q9:",len(df[(df["Genre"] == "jazz") &(df["Stem"] == "drums") &(df["Max_Silence_Sec"] >= 10)]))

    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# --- RESULTS ANALYSIS ---

# ------------------- write your code here -------------------------------
#-------------------------------------------------------------------------
# Hint: Create a pivot Table: Count by Genre vs Stem


Q4: 680
Q5: 304
Q6: 12.59
Q7: 24
Q8: 16
Q9: 7


In [40]:
stems_audio = []
try:
    # ------------------- write your code here -------------------------------
    # Load audio (Duration 5.0s for speed/consistency)
    #-------------------------------------------------------------------------
    for key in STEM_KEYS:
        path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        y, _ = librosa.load(path, sr=SR, duration=DURATION)
        stems_audio.append(y)

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [41]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.vstack(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))

#Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------


In [42]:
print("Q10:", len(mix_raw))
print("Q11:", round(rms_val,2))
print("Q12:", round(max_val,2))

Q10: 110250
Q11: 0.1
Q12: 0.59
